In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Python utilisé :", sys.executable)

Matplotlib is building the font cache; this may take a moment.


Python utilisé : c:\Users\HP\OneDrive\Documents\telecom-churn-mlops\.venv\Scripts\python.exe


In [2]:
data_path = Path("data") / "raw" / "Customer Churn.csv"

df = pd.read_csv(data_path)

print("Dimensions du dataset :", df.shape)
display(df.head())

Dimensions du dataset : (3150, 14)


,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


In [3]:
# Noms exacts des colonnes
print("Colonnes :")
print(df.columns.tolist())

# Types et valeurs manquantes
print("\nRésumé des colonnes :")
summary = pd.DataFrame({
    "type": df.dtypes,
    "valeurs_manquantes": df.isna().sum(),
    "valeurs_distinctes": df.nunique()
})
display(summary)

# Lignes entièrement identiques
print("\nNombre de lignes dupliquées :", df.duplicated().sum())

# Répartition de la cible
print("\nRépartition de Churn :")
display(
    df["Churn"].value_counts(dropna=False)
    .rename_axis("Churn")
    .to_frame("nombre_clients")
)

Colonnes :
['Call  Failure', 'Complains', 'Subscription  Length', 'Charge  Amount', 'Seconds of Use', 'Frequency of use', 'Frequency of SMS', 'Distinct Called Numbers', 'Age Group', 'Tariff Plan', 'Status', 'Age', 'Customer Value', 'Churn']

Résumé des colonnes :


,type,valeurs_manquantes,valeurs_distinctes
Call Failure,int64,0,37
Complains,int64,0,2
Subscription Length,int64,0,45
Charge Amount,int64,0,11
Seconds of Use,int64,0,1756
Frequency of use,int64,0,242
Frequency of SMS,int64,0,405
Distinct Called Numbers,int64,0,92
Age Group,int64,0,5
Tariff Plan,int64,0,2



Nombre de lignes dupliquées : 300

Répartition de Churn :


,nombre_clients
Churn,
0,2655
1,495


In [4]:
# Nettoyer uniquement les noms des colonnes
# Exemple : "Call  Failure" devient "call_failure"
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Examiner les valeurs des variables codées
for col in [
    "churn", "complains", "charge_amount",
    "tariff_plan", "status", "age", "age_group"
]:
    print(f"\n{col} :")
    print(df[col].value_counts(dropna=False).sort_index())

# Vérifier la correspondance entre Age et Age Group
print("\nCorrespondance entre age et age_group :")
display(pd.crosstab(df["age"], df["age_group"]))

# Chercher des profils identiques ayant des cibles différentes
feature_cols = df.columns.drop("churn").tolist()

labels_per_profile = (
    df.groupby(feature_cols, dropna=False)["churn"].nunique()
)

print(
    "\nNombre de profils identiques avec des labels contradictoires :",
    int((labels_per_profile > 1).sum())
)

print("\nStatistiques descriptives :")
display(df.describe().T)


churn :
churn
0    2655
1     495
Name: count, dtype: int64

complains :
complains
0    2909
1     241
Name: count, dtype: int64

charge_amount :
charge_amount
0     1768
1      617
2      395
3      199
4       76
5       30
6       11
7       14
8       19
9       14
10       7
Name: count, dtype: int64

tariff_plan :
tariff_plan
1    2905
2     245
Name: count, dtype: int64

status :
status
1    2368
2     782
Name: count, dtype: int64

age :
age
15     123
25    1037
30    1425
45     395
55     170
Name: count, dtype: int64

age_group :
age_group
1     123
2    1037
3    1425
4     395
5     170
Name: count, dtype: int64

Correspondance entre age et age_group :


age_group,1,2,3,4,5
age,,,,,
15,123,0,0,0,0
25,0,1037,0,0,0
30,0,0,1425,0,0
45,0,0,0,395,0
55,0,0,0,0,170



Nombre de profils identiques avec des labels contradictoires : 14

Statistiques descriptives :


,count,mean,std,min,25%,50%,75%,max
call_failure,3150.0,7.627937,7.263886,0.0,1.00000,6.00,12.00000,36.00
complains,3150.0,0.076508,0.265851,0.0,0.00000,0.00,0.00000,1.00
subscription_length,3150.0,32.541905,8.573482,3.0,30.00000,35.00,38.00000,47.00
charge_amount,3150.0,0.942857,1.521072,0.0,0.00000,0.00,1.00000,10.00
seconds_of_use,3150.0,4472.459683,4197.908687,0.0,1391.25000,2990.00,6478.25000,17090.00
frequency_of_use,3150.0,69.460635,57.413308,0.0,27.00000,54.00,95.00000,255.00
frequency_of_sms,3150.0,73.174921,112.237560,0.0,6.00000,21.00,87.00000,522.00
distinct_called_numbers,3150.0,23.509841,17.217337,0.0,10.00000,21.00,34.00000,97.00
age_group,3150.0,2.826032,0.892555,1.0,2.00000,3.00,3.00000,5.00
tariff_plan,3150.0,1.077778,0.267864,1.0,1.00000,1.00,1.00000,2.00


In [5]:
# df reste intact : on crée des objets séparés pour la modélisation
excluded_cols = ["churn", "age", "status", "customer_value"]

X = df.drop(columns=excluded_cols).copy()
y = df["churn"].copy()

# Même groupe pour les lignes ayant exactement les mêmes entrées.
# La cible churn n'intervient pas dans la définition des groupes.
groups = X.groupby(
    X.columns.tolist(),
    dropna=False,
    sort=False
).ngroup()

print("Variables retenues :", X.columns.tolist())
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("Nombre de profils distincts :", groups.nunique())

# Vérifications simples
assert X.index.equals(y.index)
assert X.index.equals(groups.index)
assert "churn" not in X.columns
assert not X.isna().any().any()
assert y.isin([0, 1]).all()

print("\nVérifications réussies.")

Variables retenues : ['call_failure', 'complains', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'age_group', 'tariff_plan']
Dimensions de X : (3150, 10)
Dimensions de y : (3150,)
Nombre de profils distincts : 2826

Vérifications réussies.


In [6]:
from sklearn.model_selection import StratifiedGroupKFold

# 5 partitions : on réserve la première pour le test (~20 %).
# La stratification cherche à préserver la proportion de churn,
# tout en gardant les profils identiques ensemble.
split_test = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

dev_idx, test_idx = next(split_test.split(X, y, groups))

X_dev = X.iloc[dev_idx]
y_dev = y.iloc[dev_idx]
groups_dev = groups.iloc[dev_idx]

# Parmi les ~80 % restants :
# ~60 % du total pour entraîner et ~20 % pour valider.
split_val = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    split_val.split(X_dev, y_dev, groups_dev)
)

X_train = X_dev.iloc[train_idx].copy()
y_train = y_dev.iloc[train_idx].copy()
groups_train = groups_dev.iloc[train_idx]

X_val = X_dev.iloc[val_idx].copy()
y_val = y_dev.iloc[val_idx].copy()
groups_val = groups_dev.iloc[val_idx]

X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_test = groups.iloc[test_idx]

# Vérifier qu'aucun profil ne traverse les ensembles
assert set(groups_train).isdisjoint(set(groups_val))
assert set(groups_train).isdisjoint(set(groups_test))
assert set(groups_val).isdisjoint(set(groups_test))

# Vérifier que toutes les lignes sont réparties
assert len(X_train) + len(X_val) + len(X_test) == len(X)

summary_split = pd.DataFrame({
    "ensemble": ["Entraînement", "Validation", "Test"],
    "nombre_clients": [len(y_train), len(y_val), len(y_test)],
    "taux_churn_pct": [
        100 * y_train.mean(),
        100 * y_val.mean(),
        100 * y_test.mean()
    ]
})

display(summary_split.round(2))
print("Séparation vérifiée : aucun profil partagé entre les ensembles.")

,ensemble,nombre_clients,taux_churn_pct
0,Entraînement,1883,15.61
1,Validation,632,15.82
2,Test,635,15.91


Séparation vérifiée : aucun profil partagé entre les ensembles.


In [ ]:
# Tableau d'exploration limité à l'entraînement
train_data = X_train.copy()
train_data["churn"] = y_train

# Taux de churn par catégorie, avec les effectifs
for col in ["complains", "tariff_plan", "age_group"]:
    print(f"\nChurn selon {col} :")

    table = train_data.groupby(col)["churn"].agg(
        nombre_clients="size",
        nombre_departs="sum",
        taux_churn="mean"
    )

    table["taux_churn_pct"] = 100 * table.pop("taux_churn")
    display(table.round(2))

# Comparer les usages médians
usage_cols = [
    "call_failure",
    "subscription_length",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "distinct_called_numbers"
]

print("\nValeurs médianes selon le churn :")
display(train_data.groupby("churn")[usage_cols].median().T)


Churn selon complains :


,nombre_clients,nombre_departs,taux_churn_pct
complains,,,
0,1741,173,9.94
1,142,121,85.21



Churn selon tariff_plan :


,nombre_clients,nombre_departs,taux_churn_pct
tariff_plan,,,
1,1728,289,16.72
2,155,5,3.23



Churn selon age_group :


,nombre_clients,nombre_departs,taux_churn_pct
age_group,,,
1,88,0,0.00
2,634,108,17.03
3,837,137,16.37
4,229,47,20.52
5,95,2,2.11



Valeurs médianes selon le churn :


churn,0,1
call_failure,6.0,5.0
subscription_length,35.0,34.0
seconds_of_use,3500.0,1182.5
frequency_of_use,62.0,25.0
frequency_of_sms,26.0,9.5
distinct_called_numbers,24.0,10.0


In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)

# Évaluation sur la validation uniquement
baseline_pred = baseline.predict(X_val)
baseline_scores = baseline.predict_proba(X_val)[:, 1]

baseline_results = pd.DataFrame([{
    "modele": "Baseline",
    "accuracy": accuracy_score(y_val, baseline_pred),
    "precision": precision_score(
        y_val, baseline_pred, zero_division=0
    ),
    "recall": recall_score(y_val, baseline_pred, zero_division=0),
    "f1": f1_score(y_val, baseline_pred, zero_division=0),
    "average_precision": average_precision_score(
        y_val, baseline_scores
    ),
    "roc_auc": roc_auc_score(y_val, baseline_scores)
}])

display(baseline_results.round(4))

,modele,accuracy,precision,recall,f1,average_precision,roc_auc
0,Baseline,0.8418,0.0,0.0,0.0,0.1582,0.5


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_cols = ["complains", "age_group", "tariff_plan"]
numeric_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_cols),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

logistic_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Prétraitement et modèle appris sur l'entraînement uniquement
logistic_model.fit(X_train, y_train)

# Évaluation sur la validation ; seuil initial de 0,5
logistic_scores = logistic_model.predict_proba(X_val)[:, 1]
logistic_pred = (logistic_scores >= 0.5).astype(int)

logistic_results = pd.DataFrame([{
    "modele": "Régression logistique",
    "accuracy": accuracy_score(y_val, logistic_pred),
    "precision": precision_score(
        y_val, logistic_pred, zero_division=0
    ),
    "recall": recall_score(y_val, logistic_pred, zero_division=0),
    "f1": f1_score(y_val, logistic_pred, zero_division=0),
    "average_precision": average_precision_score(
        y_val, logistic_scores
    ),
    "roc_auc": roc_auc_score(y_val, logistic_scores)
}])

display(
    pd.concat(
        [baseline_results, logistic_results],
        ignore_index=True
    ).round(4)
)

,modele,accuracy,precision,recall,f1,average_precision,roc_auc
0,Baseline,0.8418,0.000,0.00,0.0000,0.1582,0.5000
1,Régression logistique,0.9019,0.913,0.42,0.5753,0.7937,0.9418


In [10]:
from sklearn.ensemble import RandomForestClassifier

# Les arbres n'ont pas besoin de standardisation.
# Les variables catégorielles restent encodées.
forest_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_cols),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

forest_model = Pipeline(steps=[
    ("preprocessing", forest_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

forest_model.fit(X_train, y_train)

forest_scores = forest_model.predict_proba(X_val)[:, 1]
forest_pred = (forest_scores >= 0.5).astype(int)

forest_results = pd.DataFrame([{
    "modele": "Random Forest",
    "accuracy": accuracy_score(y_val, forest_pred),
    "precision": precision_score(
        y_val, forest_pred, zero_division=0
    ),
    "recall": recall_score(y_val, forest_pred, zero_division=0),
    "f1": f1_score(y_val, forest_pred, zero_division=0),
    "average_precision": average_precision_score(
        y_val, forest_scores
    ),
    "roc_auc": roc_auc_score(y_val, forest_scores)
}])

comparison = pd.concat(
    [baseline_results, logistic_results, forest_results],
    ignore_index=True
)

display(comparison.round(4))

,modele,accuracy,precision,recall,f1,average_precision,roc_auc
0,Baseline,0.8418,0.0000,0.00,0.0000,0.1582,0.5000
1,Régression logistique,0.9019,0.9130,0.42,0.5753,0.7937,0.9418
2,Random Forest,0.9209,0.8906,0.57,0.6951,0.9136,0.9849


In [11]:
from sklearn.metrics import confusion_matrix

threshold_results = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    predictions = (forest_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "seuil": threshold,
        "clients_signales": int(predictions.sum()),
        "departs_detectes": int(tp),
        "departs_manques": int(fn),
        "fausses_alertes": int(fp),
        "precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_val, predictions, zero_division=0
        )
    })

threshold_table = pd.DataFrame(threshold_results)
display(threshold_table.round(4))

,seuil,clients_signales,departs_detectes,departs_manques,fausses_alertes,precision,recall,f1
0,0.1,213,100,0,113,0.4695,1.00,0.6390
1,0.2,134,96,4,38,0.7164,0.96,0.8205
2,0.3,108,90,10,18,0.8333,0.90,0.8654
3,0.4,86,74,26,12,0.8605,0.74,0.7957
4,0.5,64,57,43,7,0.8906,0.57,0.6951


In [12]:
# Choix figés à partir de la validation
selected_model = forest_model
selected_threshold = 0.3

# Évaluation finale sur le test
test_scores = selected_model.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= selected_threshold).astype(int)

test_results = pd.DataFrame([{
    "modele": "Random Forest",
    "seuil": selected_threshold,
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(
        y_test, test_pred, zero_division=0
    ),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "average_precision": average_precision_score(
        y_test, test_scores
    ),
    "roc_auc": roc_auc_score(y_test, test_scores)
}])

display(test_results.round(4))

test_confusion = pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=[0, 1]),
    index=["Réel : reste", "Réel : départ"],
    columns=["Prédit : reste", "Prédit : départ"]
)

print("\nMatrice de confusion sur le test :")
display(test_confusion)

,modele,seuil,accuracy,precision,recall,f1,average_precision,roc_auc
0,Random Forest,0.3,0.9402,0.7647,0.901,0.8273,0.8958,0.9806



Matrice de confusion sur le test :


,Prédit : reste,Prédit : départ
Réel : reste,506,28
Réel : départ,10,91


In [13]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("telecom-churn")

print("Version MLflow :", mlflow.__version__)
print("Serveur :", mlflow.get_tracking_uri())
print("Expérience : telecom-churn")

2026/11/11 16:26:35 INFO mlflow.tracking.fluent: Experiment with name 'telecom-churn' does not exist. Creating a new experiment.


Version MLflow : 3.18.0
Serveur : http://127.0.0.1:5000
Expérience : telecom-churn


In [17]:
import hashlib
import platform

import sklearn
import mlflow
import mlflow.sklearn

# Connexion au serveur MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("telecom-churn")

# Empreinte du fichier de données
dataset_sha256 = hashlib.sha256(
    data_path.read_bytes()
).hexdigest()

experiments = [
    ("baseline", baseline, baseline_results),
    ("regression_logistique", logistic_model, logistic_results),
    ("random_forest", forest_model, forest_results),
]

metric_names = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "average_precision",
    "roc_auc",
]

run_ids = {}
model_uris = {}

for name, model, results in experiments:
    with mlflow.start_run(run_name=name) as run:
        run_ids[name] = run.info.run_id

        # Informations sur l'expérience
        mlflow.set_tags({
            "dataset": "UCI Iranian Churn",
            "logging_mode": "post_hoc",
            "split_method": "StratifiedGroupKFold",
            "selected_candidate": str(name == "random_forest"),
        })

        # Paramètres du protocole et versions
        mlflow.log_params({
            "dataset_sha256": dataset_sha256,
            "split_random_state": 42,
            "n_train": len(X_train),
            "n_validation": len(X_val),
            "n_test": len(X_test),
            "n_features": X_train.shape[1],
            "comparison_threshold": 0.5,
            "python_version": platform.python_version(),
            "sklearn_version": sklearn.__version__,
            "mlflow_version": mlflow.__version__,
        })

        # Extraire le classifieur du pipeline si nécessaire
        estimator = (
            model.named_steps["classifier"]
            if hasattr(model, "named_steps")
            else model
        )

        # Préfixer les paramètres pour éviter les conflits
        mlflow.log_params({
            f"model_{key}": value
            for key, value in estimator.get_params(
                deep=False
            ).items()
        })

        # Métriques de validation au seuil initial de 0,5
        row = results.iloc[0]

        mlflow.log_metrics({
            f"val_{metric}": float(row[metric])
            for metric in metric_names
        })

        # Variables utilisées et séparation exacte des données
        mlflow.log_dict(
            {
                "features": X_train.columns.tolist(),
                "excluded_columns": excluded_cols,
                "train_indices": X_train.index.tolist(),
                "validation_indices": X_val.index.tolist(),
                "test_indices": X_test.index.tolist(),
            },
            "data_configuration.json",
        )

        # Autoriser uniquement le type interne nécessaire aux arbres.
        # Le modèle provient de notre propre entraînement.
        trusted_types = (
            ["sklearn.tree._tree.Tree"]
            if name == "random_forest"
            else []
        )

        # Enregistrer le modèle complet, prétraitement inclus
        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            input_example=X_train.head(3),
            skops_trusted_types=trusted_types,
        )

        model_uris[name] = model_info.model_uri

        # Informations supplémentaires du modèle retenu
        if name == "random_forest":
            mlflow.log_param(
                "selected_threshold",
                selected_threshold,
            )

            # Règle de décision à appliquer aux scores
            mlflow.log_dict(
                {
                    "threshold": selected_threshold,
                    "selection_rule": (
                        "Maximum validation F1 among thresholds "
                        "0.1, 0.2, 0.3, 0.4, 0.5"
                    ),
                    "prediction_rule": (
                        "predict_proba(X)[:, 1] >= threshold"
                    ),
                },
                "decision_policy.json",
            )

            # Comparaison des seuils sur la validation
            mlflow.log_table(
                threshold_table,
                artifact_file="validation_thresholds.json",
            )

            selected_val_row = threshold_table.loc[
                threshold_table["seuil"].eq(
                    selected_threshold
                )
            ].iloc[0]

            # Métriques de validation au seuil retenu
            mlflow.log_metrics({
                f"val_selected_{metric}": float(
                    selected_val_row[metric]
                )
                for metric in ["precision", "recall", "f1"]
            })

            # Métriques de test déjà calculées
            mlflow.log_metrics({
                f"test_{metric}": float(
                    test_results.iloc[0][metric]
                )
                for metric in metric_names
            })

            # Matrice de confusion sur le test
            mlflow.log_table(
                test_confusion.reset_index().rename(
                    columns={"index": "classe_reelle"}
                ),
                artifact_file="test_confusion_matrix.json",
            )

        print(f"Expérience enregistrée : {name}")
        print(f"Run ID : {run.info.run_id}")
        print(f"Modèle : {model_info.model_uri}\n")

print("Les trois expériences sont enregistrées.")

c:\Users\HP\OneDrive\Documents\telecom-churn-mlops\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Expérience enregistrée : baseline
Run ID : 204af52faeda471595dceefac67df57b
Modèle : models:/m-71c6f8ea8305451da5722986cf3bbad3

🏃 View run baseline at: http://127.0.0.1:5000/#/experiments/1/runs/204af52faeda471595dceefac67df57b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


c:\Users\HP\OneDrive\Documents\telecom-churn-mlops\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\HP\OneDrive\Documents\telecom-churn-mlops\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missi

Expérience enregistrée : regression_logistique
Run ID : 1b694c923128419396154e0c02425c20
Modèle : models:/m-9a9900f1add9496ca3cce937ff750707

🏃 View run regression_logistique at: http://127.0.0.1:5000/#/experiments/1/runs/1b694c923128419396154e0c02425c20
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


c:\Users\HP\OneDrive\Documents\telecom-churn-mlops\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\HP\OneDrive\Documents\telecom-churn-mlops\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missi

Expérience enregistrée : random_forest
Run ID : d2a0709a8977428f9b289ce9bf913d68
Modèle : models:/m-11aa7ac387cf430196c8f702c1c60b44

🏃 View run random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/d2a0709a8977428f9b289ce9bf913d68
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Les trois expériences sont enregistrées.


In [18]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000")

model_name = "telecom-churn-random-forest"

# Modèle sauvegardé lors du run réussi
source_uri = "models:/m-11aa7ac387cf430196c8f702c1c60b44"

registered_version = mlflow.register_model(
    model_uri=source_uri,
    name=model_name,
)

client = MlflowClient()

# Associer le seuil à cette version précise
client.set_model_version_tag(
    name=model_name,
    version=registered_version.version,
    key="decision_threshold",
    value="0.3",
)

client.set_model_version_tag(
    name=model_name,
    version=registered_version.version,
    key="status",
    value="candidate",
)

# Alias lisible pour retrouver la version candidate
client.set_registered_model_alias(
    name=model_name,
    alias="candidate",
    version=registered_version.version,
)

print("Nom du modèle :", model_name)
print("Version :", registered_version.version)
print("Seuil de décision : 0.3")
print("URI candidate :", f"models:/{model_name}@candidate")

Successfully registered model 'telecom-churn-random-forest'.
2026/11/11 17:20:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: telecom-churn-random-forest, version 1
Created version '1' of model 'telecom-churn-random-forest'.


Nom du modèle : telecom-churn-random-forest
Version : 1
Seuil de décision : 0.3
URI candidate : models:/telecom-churn-random-forest@candidate


In [19]:
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = MlflowClient()
model_name = "telecom-churn-random-forest"

# Résoudre l'alias, puis charger cette version précise
candidate_version = client.get_model_version_by_alias(
    name=model_name,
    alias="candidate",
)

candidate_uri = (
    f"models:/{model_name}/{candidate_version.version}"
)

loaded_model = mlflow.sklearn.load_model(candidate_uri)

# Récupérer le seuil associé à la même version
loaded_threshold = float(
    candidate_version.tags["decision_threshold"]
)

# Comparer sur la validation, sans réentraîner
original_scores = forest_model.predict_proba(X_val)[:, 1]
loaded_scores = loaded_model.predict_proba(X_val)[:, 1]

np.testing.assert_allclose(
    loaded_scores,
    original_scores,
    rtol=1e-12,
    atol=1e-12,
)

original_predictions = (
    original_scores >= selected_threshold
).astype(int)

loaded_predictions = (
    loaded_scores >= loaded_threshold
).astype(int)

np.testing.assert_array_equal(
    loaded_predictions,
    original_predictions,
)

print("Modèle rechargé :", candidate_uri)
print("Seuil récupéré :", loaded_threshold)
print("Scores identiques : vérification réussie.")
print("Décisions identiques : vérification réussie.")

Modèle rechargé : models:/telecom-churn-random-forest/1
Seuil récupéré : 0.3
Scores identiques : vérification réussie.
Décisions identiques : vérification réussie.
